In [23]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [62]:
norm_patterns = {
        'Intensity': 'Intensity',
        'iBAQ': 'iBAQ',
        'LFQ': 'LFQ intensity'
    }

In [63]:
protein_file = '2projects combined-proteinGroups-genes.xlsx'
labels_file = '2projects combined labels.xlsx'

protein_data = pd.read_excel(protein_file, engine='openpyxl')

In [49]:
def get_cancer_positions(labels_file):
    labels_data = pd.read_excel(labels_file, engine='openpyxl')
    cancer_positions = {}
    
    for patient in tqdm(labels_data['patient number'].unique(), desc="Processing patients"):
        patient_data = labels_data[labels_data['patient number'] == patient]
        tumor_lysis = patient_data[
            (patient_data['sample diagnosis'] == 'tumor') & 
            (patient_data['extraction method'] == 'lysis')
        ]
        if not tumor_lysis.empty:
            cancer_positions[patient] = tumor_lysis['position'].tolist()
    
    return cancer_positions

In [50]:
get_cancer_positions(labels_file)

Processing patients: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 35/35 [00:00<00:00, 1401.29it/s]


{12: [3, 5],
 21: [2],
 27: [2, 3],
 31: [3],
 32: [1, 3, 5],
 39: [1, 3],
 89: [3],
 90: [3],
 72: [3],
 103: [3, 5],
 107: [3],
 111: [3],
 49: [3],
 105: [3],
 16: [3],
 99: [1, 3, 5],
 109: [3],
 59: [3],
 88: [3],
 110: [3],
 45: [3],
 60: [3],
 95: [3],
 101: [3],
 54: [1]}

In [57]:
def create_relevant_tumor_dataframes(protein_data, cancer_positions):
    dfs = {}
    
    for norm_type, pattern in tqdm(norm_patterns.items(), desc="Processing normalization types"):
        batch_columns = [col for col in protein_data.columns if col.startswith(pattern)]
        lysis_columns = [col for col in batch_columns if col.endswith('L')]
        
        relevant_columns = list(protein_data.columns[:4])
        for patient, positions in tqdm(cancer_positions.items(), desc=f"Processing {norm_type} patients"):
            for col in lysis_columns:
                # Extract patient number and position
                col_parts = col.replace(f"{pattern} ", "").split('_')  # Remove prefix first
                patient_num = col_parts[0]
                try:
                    if int(patient_num) == patient and int(col_parts[1][0]) in positions:
                        relevant_columns.append(col)
                except (ValueError, IndexError):
                    continue
        
        subset_df = protein_data[relevant_columns].copy()
        subset_df = subset_df[~(subset_df.iloc[:, 4:] == 0).all(axis=1)]
        
        output_filename = f'tumor_relevant_patients_proteomics_table_{norm_type}.csv'
        subset_df.to_csv(output_filename, index=False)
        dfs[norm_type] = subset_df
        
    return dfs

In [58]:
cancer_positions = get_cancer_positions(labels_file)
dfs = create_relevant_tumor_dataframes(protein_data, cancer_positions)

for norm_type, df in dfs.items():
    print(f"\n{norm_type} tumor analysis:")
    print(f"DataFrame shape: {df.shape}")

Processing normalization types: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.07it/s]


Intensity tumor analysis:
DataFrame shape: (6634, 37)

iBAQ tumor analysis:
DataFrame shape: (6634, 37)

LFQ tumor analysis:
DataFrame shape: (6607, 37)


In [78]:
def calculate_protein_cvs(df, norm_type):
    results = []
    
    for _, row in tqdm(df.iterrows(), desc=f"Processing {norm_type} proteins"):
        identifiers = row.iloc[:4]
        measurement_cols = df.columns[4:]
        
        patient_measurements = {}
        patient_cvs = {}
        
        # Group measurements by patient
        for col in measurement_cols:
            patient = col.replace(f"{norm_patterns[norm_type]} ", "").split('_')[0]
            measurement = row[col]
            
            if patient not in patient_measurements:
                patient_measurements[patient] = []
            
            if measurement > 0:
                patient_measurements[patient].append(measurement)
        
        # Calculate CVs for patients with multiple measurements
        for patient, measurements in patient_measurements.items():
            if len(measurements) > 1:
                cv = np.std(measurements) / np.mean(measurements)
                patient_cvs[patient] = cv
        
        # Calculate average CV
        avg_cv = np.mean(list(patient_cvs.values())) if patient_cvs else np.nan
        
        results.append({
            **dict(zip(df.columns[:4], identifiers)),
            'Patient CVs': patient_cvs,
            'Average CV': avg_cv
        })
    
    return pd.DataFrame(results)

In [79]:
def select_top_20_proteins(analysis_df):
    filtered_df = analysis_df[analysis_df['Patient CVs'].apply(len) >= 2]
    return filtered_df.sort_values('Average CV').head(20)

In [80]:
for norm_type, df in dfs.items():
    analysis_df = calculate_protein_cvs(df, norm_type)
    filtered_df = analysis_df[analysis_df['Patient CVs'].apply(len) >= 2]
    top_20_df = select_top_20_proteins(analysis_df)

    analysis_df.to_csv(f'tumor_protein_analysis_{norm_type}.csv', index=False)
    top_20_df.to_csv(f'tumor_top_20_proteins_{norm_type}.csv', index=False)
    print(f"\n{norm_type} tumor analysis:")
    print(f"Total proteins analyzed: {len(analysis_df)}")
    print(f"Proteins with sufficient measurements: {len(filtered_df)}")

Processing Intensity proteins: 6634it [00:02, 2852.33it/s]



Intensity tumor analysis:
Total proteins analyzed: 6634
Proteins with sufficient measurements: 4463


Processing iBAQ proteins: 6634it [00:02, 3030.47it/s]



iBAQ tumor analysis:
Total proteins analyzed: 6634
Proteins with sufficient measurements: 4463


Processing LFQ proteins: 6607it [00:02, 2989.41it/s]



LFQ tumor analysis:
Total proteins analyzed: 6607
Proteins with sufficient measurements: 4433
